# Huấn luyện trích xuất từ khóa — đề tài 7 (CTH625)

Notebook này là file train/fine-tune theo yêu cầu đồ án.

- **Local:** mở từ thư mục gốc repo, chạy lần lượt các cell.
- **Colab:** notebook tự nhận môi trường; chỉnh `COLAB_SOURCE` ở cell cấu hình.

Việc train thật: fit TF-IDF trên `data/corpus/`. KeyBERT dùng pretrained `vietnamese-bi-encoder` (đề bài không giao tập từ khóa vàng nên không fine-tune encoder tại đây).


In [ ]:
from pathlib import Path
import sys

def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = in_colab()
COLAB_SOURCE = "drive"  # "drive" hoặc "clone"
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/CTH625_Nhom3"
REPO_URL = ""  # điền URL git nếu COLAB_SOURCE = "clone"

print("IN_COLAB =", IN_COLAB)
print("COLAB_SOURCE =", COLAB_SOURCE)


## Cài gói (chỉ Google Colab)

Local: dùng `pip install -r requirements-train.txt` trong terminal, không chạy cell này.


In [ ]:
if IN_COLAB:
    %pip install -q underthesea scikit-learn sentence-transformers keybert pypdf chromadb huggingface_hub python-dotenv joblib
else:
    print("Local: bỏ qua pip. Cài bằng requirements-train.txt")


## Trỏ vào thư mục repo

- Local: đi ngược từ `notebooks/` đến thư mục có `src/`.
- Colab + Drive: mount Drive, `DRIVE_PROJECT_PATH` phải chứa `src/` và `data/`.
- Colab + clone: cần `REPO_URL` (chưa có thì để trống và upload repo thủ công).


In [ ]:
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Không thấy src/pipeline.py. Hãy chạy notebook từ repo hoặc mount đúng Drive.")

if IN_COLAB:
    if COLAB_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        project_root = Path(DRIVE_PROJECT_PATH)
        if not (project_root / "src" / "pipeline.py").exists():
            raise FileNotFoundError(f"Drive không có repo tại {project_root}")
    elif COLAB_SOURCE == "clone":
        if not REPO_URL:
            raise ValueError("Điền REPO_URL trước khi clone.")
        import subprocess
        subprocess.check_call(["git", "clone", REPO_URL, "/content/CTH625_Nhom3"])
        project_root = Path("/content/CTH625_Nhom3")
    else:
        raise ValueError('COLAB_SOURCE phải là "drive" hoặc "clone"')
else:
    project_root = find_project_root()

import os
os.chdir(project_root)
sys.path.insert(0, str(project_root))
print("PROJECT_ROOT =", project_root)


In [ ]:
from src.config import CORPUS_DIR, EMBEDDING_MODEL, TFIDF_VECTORIZER_PATH
from src.io_text import list_text_files, read_path
from src.keywords_tfidf import extract_tfidf, fit_tfidf, load_tfidf
from src.pipeline import run
from src.preprocess import tokenize_words

corpus_files = list_text_files(CORPUS_DIR)
if not corpus_files:
    raise FileNotFoundError(f"Không có .txt trong {CORPUS_DIR}")

corpus_texts = [read_path(path) for path in corpus_files]
print(f"Số văn bản: {len(corpus_files)}")
for path, text in zip(corpus_files[:5], corpus_texts[:5]):
    print(f"{path.relative_to(CORPUS_DIR)}: {len(text)} ký tự, {len(tokenize_words(text))} token")
if len(corpus_files) > 5:
    print("...")


## Fit TF-IDF (phương pháp 1)

Mỗi file corpus là một văn bản. Token đưa vào vectorizer đã lọc stop words và chỉ giữ từ loại N/V/A.


In [ ]:
vectorizer = fit_tfidf(corpus_texts, save_path=TFIDF_VECTORIZER_PATH)
print("Đã lưu:", TFIDF_VECTORIZER_PATH)
print("Số term trong vocabulary:", len(vectorizer.get_feature_names_out()))

sample = corpus_texts[0]
print("\nTừ khóa TF-IDF trên", corpus_files[0].name)
for term, score in extract_tfidf(sample, top_n=8, vectorizer=vectorizer):
    print(f"  {score:.4f}  {term}")


## KeyBERT + vietnamese-bi-encoder (phương pháp 2)

Cell này tải encoder từ Hugging Face. Trên Colab nên chọn GPU. Local CPU vẫn chạy được, chậm hơn.


In [ ]:
from src.keywords_keybert import extract_keybert

print("Embedding model:", EMBEDDING_MODEL)
print("Từ khóa KeyBERT trên", corpus_files[0].name)
for term, score in extract_keybert(sample, top_n=8):
    print(f"  {score:.4f}  {term}")


## Gọi cùng pipeline với web

`persist=False` để không ghi Chroma khi đang thử nghiệm. Tóm tắt LLM chỉ chạy nếu đã cấu hình `HF_TOKEN`.


In [ ]:
demo = run(
    sample,
    method="tfidf",
    top_n=8,
    do_summary=False,
    persist=False,
    source=corpus_files[0].name,
)
print("method:", demo.method)
print("N/V/A:", ", ".join(f"{t}/{p}" for t, p in demo.content_words[:20]))
print("keywords:", demo.keywords)

loaded = load_tfidf()
print("Nạp lại vectorizer:", loaded is not None)


## Ghi nhớ

- Web nạp `models/tfidf_vectorizer.joblib` nếu file tồn tại.
- Trên Colab + Drive: copy `models/` về máy trước khi deploy Streamlit nếu muốn Cloud dùng đúng IDF đã fit.
